**Cell 1: Download Dataset & Basic Imports**

In [10]:
import kagglehub
import os
import pandas as pd
import numpy as np

# Download the dataset (run only once)
path = kagglehub.dataset_download("ziya07/elderly-fall-detection-iot-dataset")
print("Dataset path:", path)

# List files
print("Files in dataset:", os.listdir(path))

Using Colab cache for faster access to the 'elderly-fall-detection-iot-dataset' dataset.
Dataset path: /kaggle/input/elderly-fall-detection-iot-dataset
Files in dataset: ['fall_detection.csv', 'archive (14)']


**Cell 2: Load Data & Initial Exploration**

In [11]:
def load_and_explore(dataset_path):
    all_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.csv')]
    dfs = []
    for file in all_files:
        df = pd.read_csv(file)
        print(f"\n=== {os.path.basename(file)} ===")
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print("Unique labels:", df['label'].unique())
        dfs.append(df)

    data = pd.concat(dfs, ignore_index=True)
    print("\n=== Full Dataset ===")
    print("Total rows:", len(data))
    print("Label distribution (original):")
    print(data['label'].value_counts())
    return data

data = load_and_explore(path)
data.head()


=== fall_detection.csv ===
Shape: (25000, 15)
Columns: ['sequence_id', 'timestep', 'accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z', 'pitch', 'roll', 'yaw', 'floor_vibration', 'room_occupancy', 'pressure_mat', 'label']
Unique labels: ['fall_forward' 'fall_side_left' 'fall_slump' 'lie_down' 'walk' 'bend'
 'fall_backward' 'fall_side_right' 'sit' 'stand']

=== Full Dataset ===
Total rows: 25000
Label distribution (original):
label
fall_backward      2950
fall_forward       2850
sit                2800
stand              2700
bend               2700
walk               2700
fall_side_right    2350
lie_down           2350
fall_side_left     1850
fall_slump         1750
Name: count, dtype: int64


,sequence_id,timestep,accel_x,accel_y,accel_z,gyro_x,gyro_y,gyro_z,pitch,roll,yaw,floor_vibration,room_occupancy,pressure_mat,label
0,0,0,-0.259997,-0.279199,-0.130207,0.090208,0.002520,0.122536,-0.252758,0.637356,-1.182670,0.0,0.0,0.0,fall_forward
1,0,1,0.274314,0.167319,0.013184,-0.003681,0.020015,-0.025951,-1.676197,0.586527,0.903426,0.0,0.0,0.0,fall_forward
2,0,2,-0.003041,0.046530,0.389528,0.076178,0.002519,0.068852,-0.262802,2.196539,-0.124426,0.0,0.0,0.0,fall_forward
3,0,3,-0.301860,-0.095863,-0.067154,0.058485,0.100037,-0.166963,-0.644498,0.343699,1.520753,0.0,0.0,0.0,fall_forward
4,0,4,-0.224695,0.047103,-0.126238,-0.239143,-0.093766,0.165844,-0.647303,-1.019149,-0.007794,0.0,0.0,0.0,fall_forward


**Cell 3: Comprehensive Preprocessing**

In [12]:
from sklearn.preprocessing import StandardScaler

# 1. Binary label mapping (fall vs non-fall)
data['label'] = data['label'].apply(lambda x: 1 if 'fall' in str(x).lower() else 0)

# 2. Handle missing values
data = data.fillna(method='ffill').fillna(0)  # forward fill, then 0 for remaining

# 3. Select features (raw sensor data - best for deep learning)
feature_cols = ['accel_x', 'accel_y', 'accel_z', 'gyro_x', 'gyro_y', 'gyro_z']

# Optional: Add magnitude features for better robustness
data['accel_magnitude'] = np.sqrt(data['accel_x']**2 + data['accel_y']**2 + data['accel_z']**2)
data['gyro_magnitude'] = np.sqrt(data['gyro_x']**2 + data['gyro_y']**2 + data['gyro_z']**2)
feature_cols += ['accel_magnitude', 'gyro_magnitude']  # Now 8 channels

# 4. Sliding window creation
WINDOW_SIZE = 200
STEP_SIZE = 50

def create_windows(df, features):
    X, y = [], []
    for i in range(0, len(df) - WINDOW_SIZE, STEP_SIZE):
        window = df[features].iloc[i:i+WINDOW_SIZE].values
        label = df['label'].iloc[i:i+WINDOW_SIZE].max()  # 1 if any fall in window
        X.append(window)
        y.append(label)
    return np.array(X), np.array(y)

X, y = create_windows(data, feature_cols)
print("Windows shape:", X.shape)  # (samples, 200, 8)
print("Label distribution in windows:", np.bincount(y))

# 5. Per-window standardization (critical for sensor drift)
scaler = StandardScaler()
X_normalized = np.array([
    scaler.fit_transform(window) for window in X
])

print("Preprocessing complete!")
print("Final X shape:", X_normalized.shape)

/tmp/ipython-input-3736195164.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill').fillna(0)  # forward fill, then 0 for remaining


Windows shape: (496, 200, 8)
Label distribution in windows: [ 44 452]
Preprocessing complete!
Final X shape: (496, 200, 8)


**Cell 4: Deep Learning - GRU Model in PyTorch**

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y, test_size=0.2, stratify=y, random_state=42
)

# PyTorch Dataset
class FallDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(FallDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(FallDataset(X_test, y_test), batch_size=64, shuffle=False)

# GRU Model
class GRUModel(nn.Module):
    def __init__(self, input_size=8, hidden_size=128, num_layers=2, num_classes=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers,
                          batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        out, h_n = self.gru(x)
        out = self.dropout(h_n[-1])
        return self.fc(out)

model = GRUModel(input_size=X_normalized.shape[2])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Class weights for imbalance
# Fix: Explicitly set dtype to torch.float32 to match model outputs
weights = torch.tensor([1.0, len(y_train)/np.sum(y_train)], dtype=torch.float32) if np.sum(y_train) > 0 else None
criterion = nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("GRU Model ready on", device)


GRU Model ready on cpu


**Cell 5: Training the GRU Model**

In [16]:
# Training loop
model.train()
for epoch in range(50):
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 | Avg Loss: {total_loss/len(train_loader):.4f}")

print("Training completed!")

Epoch 10/50 | Avg Loss: 0.0576
Epoch 20/50 | Avg Loss: 0.0006
Epoch 30/50 | Avg Loss: 0.0003
Epoch 40/50 | Avg Loss: 0.0002
Epoch 50/50 | Avg Loss: 0.0001
Training completed!


**Cell 6: Evaluation & Alert Simulation**

In [38]:
model.eval()
all_probs, all_preds, all_labels = [], [], []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Non-Fall', 'Fall']))

# Simulated real-time alerts
print("\n=== Real-Time Fall Alerts (first 15 samples) ===")
for i in range(min(15, len(all_probs))):
    prob = all_probs[i]
    true = "Fall" if all_labels[i] == 1 else "Non-Fall"
    if prob > 0.8:
        alert = "Fall detected! Sending alert message."
    elif prob > 0.5:
        alert = "Likely to fall. Be cautious."
    else:
        alert = "No fall risk."
    print(f"Sample {i+1} | True: {true} | Prob: {prob:.2f} → {alert}")

# Save the trained model
torch.save(model.state_dict(), "elderly_fall_gru_model.pth")

Confusion Matrix:
[[ 1  8]
 [ 0 91]]

Classification Report:
              precision    recall  f1-score   support

    Non-Fall       1.00      0.11      0.20         9
        Fall       0.92      1.00      0.96        91

    accuracy                           0.92       100
   macro avg       0.96      0.56      0.58       100
weighted avg       0.93      0.92      0.89       100


=== Real-Time Fall Alerts (first 15 samples) ===
Sample 1 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 2 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 3 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 4 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 5 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 6 | True: Fall | Prob: 0.52 → Likely to fall. Be cautious.
Sample 7 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 8 | True: Fall | Prob: 0.51 → Likely to fall. Be cautious.
Sample 9 | True: Fall | Prob: 0.50

**Cell 7: Visualization with Plotly - Confusion Matrix & Probability Distribution**

In [39]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, all_preds)
fig1 = px.imshow(cm,
                 labels=dict(x="Predicted", y="Actual", color="Count"),
                 x=['Non-Fall', 'Fall'],
                 y=['Non-Fall', 'Fall'],
                 text_auto=True,
                 color_continuous_scale='Blues')
fig1.update_layout(title="Confusion Matrix")
fig1.show()

# 2. Fall Probability Distribution
fig2 = px.histogram(all_probs, nbins=50, color=all_labels,
                    labels={'value': 'Fall Probability', 'color': 'True Label'},
                    color_discrete_map={0: 'blue', 1: 'red'},
                    title="Distribution of Predicted Fall Probabilities",
                    opacity=0.7)
fig2.update_layout(barmode='overlay')
fig2.show()

In [40]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Use subset for fast comparison (or full data if you have GPU/time)
# Reduce to 5000 samples if dataset is large
subset_size = min(5000, len(X_normalized))
indices = np.random.choice(len(X_normalized), subset_size, replace=False)
X_sub = X_normalized[indices]
y_sub = y[indices]

# DataLoader
dataset = TensorDataset(torch.tensor(X_sub, dtype=torch.float32),
                        torch.tensor(y_sub, dtype=torch.long))
loader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Using {subset_size} samples for comparison")

Using device: cpu
Using 496 samples for comparison


In [41]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=8, hidden_size=128, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

class GRUModel(nn.Module):
    def __init__(self, input_size=8, hidden_size=128, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        _, h_n = self.gru(x)
        return self.fc(h_n[-1])

class CNNLSTMModel(nn.Module):
    def __init__(self, input_size=8):
        super().__init__()
        self.conv1 = nn.Conv1d(input_size, 64, 5, padding=2)
        self.conv2 = nn.Conv1d(64, 128, 5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(128, 128, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(128, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.transpose(1, 2)
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

models_to_compare = {
    'GRU (Optimized)': GRUModel().to(device),
    'LSTM': LSTMModel().to(device),
    'CNN-LSTM (Heavier)': CNNLSTMModel().to(device)
}

comparison_results = {}

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:123: UserWarning:

dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1



In [42]:
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
import time
from sklearn.metrics import accuracy_score

# Use ONLY 1000 samples max for super-fast comparison
subset_size = min(1000, len(X_normalized))  # Very small
indices = np.random.choice(len(X_normalized), subset_size, replace=False)
X_sub = X_normalized[indices]
y_sub = y[indices]

# Smaller batch size for speed
loader = DataLoader(TensorDataset(torch.tensor(X_sub, dtype=torch.float32),
                                  torch.tensor(y_sub, dtype=torch.long)),
                    batch_size=32, shuffle=True)

print(f"Using {subset_size} samples ")

Using 496 samples 


In [43]:
criterion = nn.CrossEntropyLoss()

for name, model in models_to_compare.items():
    print(f"\n=== Training {name} ===")
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()

    start_time = time.time()

    for epoch in range(10):  # Reduced from 20 → 10
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"   Epoch {epoch+1}/10 done")  # Progress tracker

    training_time = time.time() - start_time
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Eval
    model.eval()
    with torch.no_grad():
        test_inputs, test_labels = next(iter(loader))
        test_inputs = test_inputs.to(device)
        preds = model(test_inputs).argmax(dim=1).cpu().numpy()
        true = test_labels.numpy()
        acc = accuracy_score(true, preds)

    comparison_results[name] = {
        'Training Time (10 epochs)': f"{training_time:.1f}s",
        'Parameters': f"{params:,}",
        'Approx Acc (last batch)': f"{acc:.3f}"
    }


=== Training GRU (Optimized) ===
   Epoch 1/10 done
   Epoch 2/10 done
   Epoch 3/10 done
   Epoch 4/10 done
   Epoch 5/10 done
   Epoch 6/10 done
   Epoch 7/10 done
   Epoch 8/10 done
   Epoch 9/10 done
   Epoch 10/10 done

=== Training LSTM ===
   Epoch 1/10 done
   Epoch 2/10 done
   Epoch 3/10 done
   Epoch 4/10 done
   Epoch 5/10 done
   Epoch 6/10 done
   Epoch 7/10 done
   Epoch 8/10 done
   Epoch 9/10 done
   Epoch 10/10 done

=== Training CNN-LSTM (Heavier) ===
   Epoch 1/10 done
   Epoch 2/10 done
   Epoch 3/10 done
   Epoch 4/10 done
   Epoch 5/10 done
   Epoch 6/10 done
   Epoch 7/10 done
   Epoch 8/10 done
   Epoch 9/10 done
   Epoch 10/10 done


In [44]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Extract lightweight features
def extract_quick_features(X):
    # Fix: Remove the extra np.array([...]) wrapper that was adding an unnecessary dimension.
    # The concatenated array already has the correct number of samples.
    return np.concatenate([
        np.mean(X, axis=1),
        np.std(X, axis=1),
        np.max(X, axis=1),
        np.min(X, axis=1)
    ], axis=1)

X_features = extract_quick_features(X_sub)
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_features, y_sub, test_size=0.2, random_state=42)

start_time = time.time()
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_f, y_train_f)
training_time = time.time() - start_time

preds_rf = rf.predict(X_test_f)
acc_rf = accuracy_score(y_test_f, preds_rf)
f1_rf = f1_score(y_test_f, preds_rf)

print("\n=== Classical ML: Random Forest ===")
print(f"Training Time: {training_time:.2f}s")
print(f"Test Accuracy: {acc_rf:.3f}")
print(f"Test F1-Score: {f1_rf:.3f}")
print(f"Feature vector size: {X_features.shape[1]} (very lightweight)")


=== Classical ML: Random Forest ===
Training Time: 0.52s
Test Accuracy: 1.000
Test F1-Score: 1.000
Feature vector size: 32 (very lightweight)


In [45]:
print("\n=== FAST MODEL COMPARISON RESULTS ===")
for k, v in comparison_results.items():
    print(f"{k}:")
    for mk, mv in v.items():
        print(f"   {mk}: {mv}")




=== FAST MODEL COMPARISON RESULTS ===
GRU (Optimized):
   Training Time (10 epochs): 105.8s
   Parameters: 152,322
   Approx Acc (last batch): 0.906
LSTM:
   Training Time (10 epochs): 80.7s
   Parameters: 203,010
   Approx Acc (last batch): 1.000
CNN-LSTM (Heavier):
   Training Time (10 epochs): 11.1s
   Parameters: 176,066
   Approx Acc (last batch): 1.000
